# Module 5: CLI Interface - Part 1

## 🎯 **Module Overview**
This module documents the `cli.py` file - the command-line interface for the SpecCoder agent. The CLI provides user-friendly access to all SpecCoder functionality including code generation, testing, healing, organization, and pipeline orchestration.

## 📋 **What We'll Learn**
- Argument parsing and command routing
- Import handling for different execution contexts
- Error handling and user feedback
- Pipeline stage execution through CLI
- Logging configuration and verbose output

## 🔧 **Key Components**
- **Command Functions**: Individual handlers for each CLI command
- **Argument Parser**: Configures command-line arguments and subcommands
- **Import Handling**: Manages relative vs absolute imports
- **Error Handling**: Provides user-friendly error messages
- **Logging Setup**: Configures verbose and normal output modes

In [ ]:
# Cell 1: Setup and Mock Dependencies
import sys
import logging
from pathlib import Path
from unittest.mock import Mock, MagicMock, patch
from typing import Optional
import argparse

# Mock the external dependencies
mock_generator = Mock()
mock_tester = Mock()
mock_healer = Mock()
mock_organizer = Mock()
mock_orchestrator = Mock()

# Create mock result objects
mock_generation_result = Mock()
mock_generation_result.success = True
mock_generation_result.files_generated = ['file1.py', 'file2.py']
mock_generation_result.execution_time = 2.5
mock_generation_result.warnings = ['Warning 1']
mock_generation_result.errors = []

mock_generator.generate_from_spec.return_value = mock_generation_result
mock_tester.run_tests.return_value = True
mock_healer.heal_all_functions.return_value = [Mock(success=True)]
mock_organizer.run.return_value = True
mock_orchestrator._stage1_spec_to_scaffold.return_value = True
mock_orchestrator._stage2_scaffold_to_requirements.return_value = True
mock_orchestrator._stage3_requirements_to_alignment.return_value = True
mock_orchestrator._stage4_alignment_to_code.return_value = True
mock_orchestrator.run_full_pipeline.return_value = True

# Mock the imports
sys.modules['generator'] = mock_generator
sys.modules['tester'] = mock_tester
sys.modules['healer'] = mock_healer
sys.modules['organizer'] = mock_organizer
sys.modules['orchestrator'] = mock_orchestrator

print("✅ Mock dependencies setup complete")
print(f"📁 Mock generator: {mock_generator}")
print(f"🧪 Mock tester: {mock_tester}")
print(f"🧩 Mock healer: {mock_healer}")
print(f"📁 Mock organizer: {mock_organizer}")
print(f"🚀 Mock orchestrator: {mock_orchestrator}")

In [ ]:
# Cell 2: Import Handling and Setup Functions
"""
Lines 8-38: Import handling and setup functions

The CLI handles imports in two ways:
1. Relative imports (when installed as package)
2. Absolute imports (when running directly)

This flexibility allows the CLI to work in different execution contexts.
"""

import sys
import argparse
import logging
from pathlib import Path

# Add current directory to path for imports (Line 14)
# This ensures the CLI can find the modules when run directly
sys.path.insert(0, str(Path(__file__).parent) if '__file__' in locals() else str(Path.cwd()))

def setup_logging(verbose: bool = False):
    """
    Lines 32-38: Setup logging configuration
    
    Configures logging based on verbose flag:
    - verbose=True: DEBUG level (detailed output)
    - verbose=False: INFO level (normal output)
    
    Format includes timestamp, logger name, level, and message.
    """
    level = logging.DEBUG if verbose else logging.INFO
    logging.basicConfig(
        level=level,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )

print("🔧 Import handling and logging setup:")
print(f"📂 Current path: {Path.cwd()}")
print(f"🐍 Python path entries: {len(sys.path)}")

# Test logging setup
print("\n📝 Testing logging setup:")
setup_logging(verbose=False)
logging.info("Info level test")
setup_logging(verbose=True)
logging.debug("Debug level test")

In [ ]:
# Cell 3: Generate Command Implementation
"""
Lines 41-78: Generate command implementation

The generate command handles code generation from OpenSpec files.
It includes comprehensive error handling and user feedback.
"""

def cmd_generate(args):
    """
    Lines 41-78: Generate code from spec
    
    Process:
    1. Setup logging based on verbose flag
    2. Validate spec file exists
    3. Create CodeGenerator instance
    4. Generate code from spec
    5. Display results with emojis for better UX
    6. Handle errors gracefully
    """
    setup_logging(args.verbose)
    
    if not args.spec_file.exists():
        print(f"Error: Specification file not found: {args.spec_file}")
        return 1
    
    try:
        generator = mock_generator.CodeGenerator()
        print(f"Generating code from: {args.spec_file}")
        result = generator.generate_from_spec(args.spec_file, args.output)
        
        if result.success:
            print(f"✅ Generation completed successfully!")
            print(f"📁 Generated {len(result.files_generated)} files:")
            for file_path in result.files_generated:
                print(f"   - {file_path}")
            print(f"⏱️  Execution time: {result.execution_time:.2f}s")
            
            if result.warnings:
                print(f"⚠️  Warnings:")
                for warning in result.warnings:
                    print(f"   - {warning}")
            return 0
        else:
            print(f"❌ Generation failed!")
            for error in result.errors:
                print(f"   - {error}")
            return 1
            
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1

# Test the generate command
print("🧪 Testing generate command:")

# Create mock args
class MockArgs:
    def __init__(self):
        self.spec_file = Path("test_spec.yaml")
        self.output = Path("output")
        self.verbose = False

# Test with existing file
args = MockArgs()
args.spec_file = Path("/tmp/test_spec.yaml")
# Create the file for testing
Path("/tmp/test_spec.yaml").touch()

result = cmd_generate(args)
print(f"\n📊 Generate command result: {result}")

# Test with non-existent file
args.spec_file = Path("nonexistent.yaml")
result = cmd_generate(args)
print(f"📊 Generate command with missing file: {result}")

In [ ]:
# Cell 4: Test, Heal, and Organize Commands
"""
Lines 80-152: Test, Heal, and Organize command implementations

These commands handle the maintenance and organization aspects:
- test: Run automated tests
- heal: Repair failing tests
- organize: Restructure code
"""

def cmd_test(args):
    """
    Lines 80-102: Run automated tests
    
    Simple command that runs the TestAnalyzer and reports success/failure.
    """
    setup_logging(args.verbose)
    
    try:
        tester = mock_tester.TestAnalyzer()
        print("🧪 Running automated tests...")
        success = tester.run_tests()
        
        if success:
            print("✅ All tests passed!")
            return 0
        else:
            print("❌ Some tests failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Test execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1


def cmd_heal(args):
    """
    Lines 104-128: Repair failing tests
    
    Uses the Healer class to automatically fix failing tests.
    """
    setup_logging(args.verbose)
    
    try:
        healer = mock_healer.Healer(healer_dir=Path.cwd())
        print("🧩 Starting healing process...")
        results = healer.heal_all_functions()
        success = all(r.success for r in results)
        
        if success:
            print("✅ Healing completed successfully!")
            return 0
        else:
            print("⚠️ Healing completed with some issues")
            return 1
            
    except Exception as e:
        print(f"❌ Healing failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1


def cmd_organize(args):
    """
    Lines 130-152: Restructure code
    
    Uses the CodeOrganizer to improve code structure.
    """
    setup_logging(args.verbose)
    
    try:
        organizer = mock_organizer.CodeOrganizer()
        print("📁 Starting code organization...")
        success = organizer.run()
        
        if success:
            print("✅ Code organization completed!")
            return 0
        else:
            print("❌ Code organization failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Organization failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1

# Test these commands
print("🧪 Testing maintenance commands:")

args = MockArgs()

test_result = cmd_test(args)
print(f"📊 Test command result: {test_result}")

heal_result = cmd_heal(args)
print(f"📊 Heal command result: {heal_result}")

organize_result = cmd_organize(args)
print(f"📊 Organize command result: {organize_result}")

In [ ]:
# Cell 5: Pipeline Stage Commands
"""
Lines 154-252: Pipeline stage command implementations

These commands provide granular access to each pipeline stage:
- stage1: OpenSpec → Tests/Scaffold
- stage2: Test/Scaffold → Logical Requirements
- stage3: Logical Requirements → Behavioral Alignment
- stage4: Behavioral Alignment → Final Code
"""

def cmd_stage1(args):
    """
    Lines 154-180: Stage 1: OpenSpec → Tests/Scaffold
    
    Converts OpenSpec YAML to test scaffold.
    """
    setup_logging(args.verbose)
    
    if not args.spec_file.exists():
        print(f"Error: Specification file not found: {args.spec_file}")
        return 1
    
    try:
        runner = mock_orchestrator.IntegrationOrchestrator()
        print("🔧 Stage 1: Converting OpenSpec to test scaffold...")
        success = runner._stage1_spec_to_scaffold(args.spec_file, args.output)
        
        if success:
            print("✅ Stage 1 completed successfully!")
            return 0
        else:
            print("❌ Stage 1 failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Stage 1 execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1


def cmd_stage2(args):
    """
    Lines 182-204: Stage 2: Test/Scaffold → Logical Requirements
    
    Extracts logical requirements from test scaffold.
    """
    setup_logging(args.verbose)
    
    try:
        runner = mock_orchestrator.IntegrationOrchestrator()
        print("🧪 Stage 2: Extracting logical requirements from test scaffold...")
        success = runner._stage2_scaffold_to_requirements(args.output)
        
        if success:
            print("✅ Stage 2 completed successfully!")
            return 0
        else:
            print("❌ Stage 2 failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Stage 2 execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1


def cmd_stage3(args):
    """
    Lines 206-228: Stage 3: Logical Requirements → Behavioral Alignment
    
    Performs behavioral alignment analysis.
    """
    setup_logging(args.verbose)
    
    try:
        runner = mock_orchestrator.IntegrationOrchestrator()
        print("🎯 Stage 3: Performing behavioral alignment...")
        success = runner._stage3_requirements_to_alignment(args.output)
        
        if success:
            print("✅ Stage 3 completed successfully!")
            return 0
        else:
            print("❌ Stage 3 failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Stage 3 execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1


def cmd_stage4(args):
    """
    Lines 230-252: Stage 4: Behavioral Alignment → Final Code
    
    Generates final code from alignment analysis.
    Supports dry-run mode for testing setup.
    """
    setup_logging(args.verbose)
    
    try:
        runner = mock_orchestrator.IntegrationOrchestrator()
        print("⚡ Stage 4: Generating final code from alignment...")
        success = runner._stage4_alignment_to_code(args.output, dry_run=args.dry_run)
        
        if success:
            print("✅ Stage 4 completed successfully!")
            return 0
        else:
            print("❌ Stage 4 failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Stage 4 execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1

# Test stage commands
print("🧪 Testing pipeline stage commands:")

args = MockArgs()
args.spec_file = Path("/tmp/test_spec.yaml")
args.dry_run = False

stage1_result = cmd_stage1(args)
print(f"📊 Stage 1 result: {stage1_result}")

stage2_result = cmd_stage2(args)
print(f"📊 Stage 2 result: {stage2_result}")

stage3_result = cmd_stage3(args)
print(f"📊 Stage 3 result: {stage3_result}")

stage4_result = cmd_stage4(args)
print(f"📊 Stage 4 result: {stage4_result}")

In [ ]:
# Cell 6: Full Run Command
"""
Lines 254-280: Full run command implementation

The full-run command executes the entire pipeline from OpenSpec to final code.
This is the main entry point for complete code generation.
"""

def cmd_full_run(args):
    """
    Lines 254-280: Run the entire pipeline
    
    Executes all 4 stages in sequence:
    1. Validate spec file exists
    2. Create IntegrationOrchestrator
    3. Run full pipeline
    4. Report results
    """
    setup_logging(args.verbose)
    
    if not args.spec_file.exists():
        print(f"Error: Specification file not found: {args.spec_file}")
        return 1
    
    try:
        runner = mock_orchestrator.IntegrationOrchestrator()
        print("🚀 Starting full pipeline...")
        success = runner.run_full_pipeline(args.spec_file, args.output)
        
        if success:
            print("✅ Full pipeline completed successfully!")
            return 0
        else:
            print("❌ Pipeline failed!")
            return 1
            
    except Exception as e:
        print(f"❌ Pipeline execution failed: {e}")
        if args.verbose:
            import traceback
            traceback.print_exc()
        return 1

# Test full run command
print("🧪 Testing full run command:")

args = MockArgs()
args.spec_file = Path("/tmp/test_spec.yaml")

full_run_result = cmd_full_run(args)
print(f"📊 Full run result: {full_run_result}")

# Test with missing file
args.spec_file = Path("missing.yaml")
full_run_result = cmd_full_run(args)
print(f"📊 Full run with missing file: {full_run_result}")

In [ ]:
# Cell 7: Argument Parser Configuration
"""
Lines 282-416: Main function and argument parser setup

The main function configures the argument parser with all commands
and their respective arguments, then routes to the appropriate handler.
"""

def create_argument_parser():
    """
    Lines 284-383: Create and configure argument parser
    
    Sets up:
    - Main parser with description
    - Global verbose flag
    - Subparsers for each command
    - Command-specific arguments
    """
    parser = argparse.ArgumentParser(
        description="OpenSpec-driven autonomous coding agent"
    )
    
    parser.add_argument(
        "-v", "--verbose",
        action="store_true",
        help="Enable verbose logging"
    )
    
    subparsers = parser.add_subparsers(dest='command', help='Available commands')
    
    # Generate command
    gen_parser = subparsers.add_parser('generate', help='Generate code from spec')
    gen_parser.add_argument(
        'spec_file',
        type=Path,
        help='Path to the OpenSpec YAML file'
    )
    gen_parser.add_argument(
        '-o', '--output',
        type=Path,
        default=None,
        help='Output directory (default: ./output)'
    )
    
    # Test command
    test_parser = subparsers.add_parser('test', help='Run automated tests')
    
    # Heal command
    heal_parser = subparsers.add_parser('heal', help='Repair failing tests')
    heal_parser.add_argument(
        '--retest',
        action='store_true',
        help='Force re-running tests even if test results already exist'
    )
    
    # Organize command
    org_parser = subparsers.add_parser('organize', help='Restructure code')
    
    # Stage commands
    stage1_parser = subparsers.add_parser('stage1', help='Stage 1: OpenSpec → Tests/Scaffold')
    stage1_parser.add_argument('spec_file', type=Path, help='Path to the OpenSpec YAML file')
    stage1_parser.add_argument('-o', '--output', type=Path, default=None, help='Output directory (default: ./output)')
    
    stage2_parser = subparsers.add_parser('stage2', help='Stage 2: Test/Scaffold → Logical Requirements')
    stage2_parser.add_argument('-o', '--output', type=Path, default=None, help='Output directory (default: ./output)')
    
    stage3_parser = subparsers.add_parser('stage3', help='Stage 3: Logical Requirements → Behavioral Alignment')
    stage3_parser.add_argument('-o', '--output', type=Path, default=None, help='Output directory (default: ./output)')
    
    stage4_parser = subparsers.add_parser('stage4', help='Stage 4: Behavioral Alignment → Final Code')
    stage4_parser.add_argument('-o', '--output', type=Path, default=None, help='Output directory (default: ./output)')
    stage4_parser.add_argument('--dry-run', action='store_true', help='Dry run mode - check setup without calling AI models')
    
    # Full run command
    full_parser = subparsers.add_parser('full-run', help='Run the entire pipeline')
    full_parser.add_argument('spec_file', type=Path, help='Path to the OpenSpec YAML file')
    full_parser.add_argument('-o', '--output', type=Path, default=None, help='Output directory (default: ./output)')
    
    return parser

def route_command(args):
    """
    Lines 390-416: Route to appropriate command handler
    
    Maps command names to their handler functions.
    """
    command_map = {
        'generate': cmd_generate,
        'test': cmd_test,
        'heal': cmd_heal,
        'organize': cmd_organize,
        'stage1': cmd_stage1,
        'stage2': cmd_stage2,
        'stage3': cmd_stage3,
        'stage4': cmd_stage4,
        'full-run': cmd_full_run,
    }
    
    if args.command in command_map:
        return command_map[args.command](args)
    else:
        print(f"Unknown command: {args.command}")
        return 1

# Test argument parser
print("🧪 Testing argument parser:")

parser = create_argument_parser()
print(f"📝 Parser created: {parser}")

# Test parsing different commands
test_args = [
    ['generate', 'test.yaml'],
    ['test'],
    ['heal'],
    ['organize'],
    ['stage1', 'test.yaml'],
    ['full-run', 'test.yaml', '-v']
]

for arg_list in test_args:
    try:
        args = parser.parse_args(arg_list)
        print(f"✅ Parsed {arg_list[0]}: command={args.command}, verbose={getattr(args, 'verbose', False)}")
    except SystemExit:
        print(f"⚠️  Failed to parse: {arg_list}")

In [ ]:
# Cell 8: CLI Integration Testing
"""
Complete CLI integration testing with all components.

This cell tests the entire CLI workflow to ensure all commands
work correctly with the argument parser and routing.
"""

def main():
    """
    Lines 282-419: Main CLI entry point
    
    Complete main function that:
    1. Creates argument parser
    2. Parses command line arguments
    3. Shows help if no command provided
    4. Routes to appropriate handler
    5. Returns exit code
    """
    parser = create_argument_parser()
    
    # For testing, we'll simulate command line arguments
    # In real usage, this would be: args = parser.parse_args()
    
    return parser

# Test complete CLI integration
print("🧪 Complete CLI Integration Testing:")
print("=" * 50)

# Initialize the CLI
cli_parser = main()
print(f"🚀 CLI initialized: {cli_parser.description}")

# Test all command workflows
print("\n📋 Testing all command workflows:")

workflows = [
    (['generate', '/tmp/test_spec.yaml'], 'Code Generation'),
    (['test'], 'Test Execution'),
    (['heal'], 'Test Healing'),
    (['organize'], 'Code Organization'),
    (['stage1', '/tmp/test_spec.yaml'], 'Stage 1'),
    (['stage2'], 'Stage 2'),
    (['stage3'], 'Stage 3'),
    (['stage4', '--dry-run'], 'Stage 4 (Dry Run)'),
    (['full-run', '/tmp/test_spec.yaml'], 'Full Pipeline')
]

for cmd_args, description in workflows:
    print(f"\n🔧 Testing {description}:")
    try:
        args = cli_parser.parse_args(cmd_args)
        result = route_command(args)
        print(f"   ✅ {description}: {result} (exit code)")
    except Exception as e:
        print(f"   ❌ {description}: {e}")

# Test error handling
print("\n🛡️  Testing error handling:")

# Test with missing spec file
try:
    args = cli_parser.parse_args(['generate', 'missing.yaml'])
    result = route_command(args)
    print(f"✅ Missing file handling: {result}")
except Exception as e:
    print(f"❌ Missing file handling failed: {e}")

# Test verbose mode
print("\n📝 Testing verbose mode:")
try:
    args = cli_parser.parse_args(['-v', 'test'])
    print(f"✅ Verbose flag parsed: verbose={args.verbose}")
    result = route_command(args)
    print(f"✅ Verbose test execution: {result}")
except Exception as e:
    print(f"❌ Verbose mode failed: {e}")

print("\n🎉 CLI Integration Testing Complete!")
print("\n📊 Summary:")
print(f"   - Total commands: 9")
print(f"   - Argument parser: ✅")
print(f"   - Command routing: ✅")
print(f"   - Error handling: ✅")
print(f"   - Verbose logging: ✅")
print(f"   - File validation: ✅")

print("\n🔍 CLI Features Documented:")
print("   📝 Argument parsing with subcommands")
print("   🔄 Flexible import handling")
print("   🛡️  Comprehensive error handling")
print("   📊 Detailed progress reporting")
print("   🎯 Stage-by-stage pipeline access")
print("   🚀 Full pipeline execution")
print("   🧪 Test and healing integration")
print("   📁 Code organization support")
print("   🔧 Dry-run mode for testing")
print("   📝 Verbose logging support")